# Single-Cell RNA-seq: From Raw Counts to Cell Types
## Module 02 — Notebook 05

## TL;DR — Plain English Introduction

**What is scRNA-seq?**  
Ordinary (bulk) RNA-seq measures the *average* gene activity across millions of cells in a tissue sample — like calculating a class average on an exam. Single-cell RNA-seq (scRNA-seq) measures gene activity in **each cell individually** — like seeing every student's score. This reveals hidden diversity: a tumour is not one cell type but dozens; a blood sample contains T-cells, B-cells, monocytes, and dendritic cells, each with a completely different gene expression programme.

**What you will build in this notebook:**
- A realistic simulated count matrix (500 cells × 2000 genes, 5 cell types)
- A full QC, normalisation, and feature selection pipeline
- PCA + t-SNE visualisation coloured by cell type
- KNN-based clustering and marker gene identification via differential expression
- A pseudotime trajectory of T-cell activation
- Connection to drug target discovery and protein ML

**Time:** ~5 hours  
**Difficulty:** ★★★☆☆  
**Prerequisites:** Module 02/02 (bulk RNA-seq), Module 00 (numpy/pandas/sklearn)

**Dependencies used:** `numpy`, `pandas`, `matplotlib`, `sklearn` only — no scanpy or anndata required.

**Primary literature:**  
- Macosko et al. 2015 (Drop-seq) doi:10.1016/j.cell.2015.05.002  
- Wolf et al. 2018 (Scanpy) doi:10.1186/s13059-017-1382-0  
- Luecken & Theis 2019 (Best practices) doi:10.15252/msb.20188746  
- McInnes et al. 2018 (UMAP) arxiv:1802.03426

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain the difference between bulk and single-cell RNA-seq and when each is appropriate
2. Apply QC filters (min/max genes per cell, mitochondrial %) to a real count matrix
3. Perform library-size normalisation and log1p transformation, explaining why both are necessary
4. Select highly variable genes using the dispersion method
5. Interpret PCA and t-SNE embeddings coloured by cell type
6. Cluster cells using KNN graphs and compare clusters to known cell types
7. Find marker genes per cluster using log fold-change and the Mann-Whitney U test
8. Trace a T-cell activation pseudotime trajectory
9. Articulate how scRNA-seq informs protein structure and drug discovery priorities

**Cross-references:**
- Module 02/02: bulk RNA-seq and DESeq2-style normalisation (prerequisite)
- Module 04/01: ML for omics — PCA, UMAP, and gene expression classifiers
- Module 09/01: learning curves and model diagnostics applied to clustering quality

## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **RNA-seq basics** — gene counts, normalization, differential expression. *Review: `02_genomics_gene_analysis/02_rnaseq_analysis.ipynb`*
- **PCA** — dimensionality reduction to visualize high-dimensional data. *Review: `04_ml_bioinformatics/01_ml_for_omics.ipynb`*
- **Python data structures** — dicts, lists, numpy arrays. *Review: `00_python_ml_basics/01_python_core_for_bioinformatics.ipynb`*

**Quick recap of terms used here:**
- **scRNA-seq** — sequencing RNA from individual cells (not bulk tissue)
- **dropout** — zeros in scRNA-seq data where a gene is expressed but not detected (technical artifact)
- **UMAP** — non-linear dimensionality reduction for visualizing cell clusters
- **Leiden clustering** — graph-based community detection for finding cell types (connects to Module 06 GNNs)

**Where this connects forward:**
- Graph-based clustering here → graph neural networks in `06_structural_ml_gnns/01_structure_ml.ipynb`
- Dimensionality reduction here → PCA/embedding in `04/01` and protein embeddings in `15/01`

> **Analogy:** Bulk RNA-seq is like blending a fruit smoothie and trying to guess the ingredients. Single-cell RNA-seq is like keeping each fruit separate -- you can see exactly which cells express which genes.

## Section 1 — Beginner Frame: Bulk vs Single-Cell RNA-seq

### The Class Average Analogy

Imagine you are a teacher grading a maths exam. **Bulk RNA-seq** is like reporting the class average — 72/100. It tells you something useful, but it hides everything interesting: the three students who scored 100, the five who scored 20, the bimodal distribution suggesting two distinct learning groups.

**Single-cell RNA-seq** gives you every student's individual score. Now you can:
- Cluster students into "struggling", "average", and "advanced" groups
- Ask: what distinguishes the top group? (marker genes)
- Trace a trajectory: which students are improving over time? (pseudotime)

### Technical Vocabulary

| Term | Plain English | Why It Matters |
|------|--------------|----------------|
| **UMI** (Unique Molecular Identifier) | A unique barcode attached to each RNA molecule *before* PCR amplification | Lets us count *original* molecules, not PCR duplicates — corrects amplification bias |
| **Cell barcode** | A short DNA sequence that identifies which droplet (cell) a read came from | Allows thousands of cells to be sequenced simultaneously in one lane |
| **Count matrix** | A cells × genes table where entry (i,j) = number of UMIs of gene j in cell i | The primary data structure for all downstream analysis |
| **Cell type** | A biologically distinct class of cell with a characteristic gene expression programme | T-cell, B-cell, monocyte, etc. — identified by marker genes |
| **Trajectory / Pseudotime** | An ordering of cells along a developmental or activation path | Reconstructs dynamic processes (e.g., naive T-cell → effector T-cell) from a static snapshot |
| **Doublet** | Two cells captured in the same droplet, producing a mixed count profile | A major technical artefact — identified by anomalously high UMI counts |
| **Library size** | Total UMI count per cell | Varies due to capture efficiency, not biology — must be normalised out |
| **HVG** | Highly Variable Gene — varies more across cells than expected by chance | The informative subset used for PCA and clustering |
| **Leiden clustering** | Graph-based community detection algorithm used to find cell clusters | Resolution parameter controls granularity — analogous to k in k-means |

### Key Technological Innovations (Literature)

The field was transformed by **Drop-seq** (Macosko et al., *Cell* 2015, doi:10.1016/j.cell.2015.05.002), which co-encapsulated single cells with barcoded beads in nanolitre droplets, enabling transcriptomic profiling of thousands of cells simultaneously. The 10x Genomics Chromium platform scaled this to ~10,000 cells/experiment. The **Scanpy** Python framework (Wolf et al., *Genome Biology* 2018, doi:10.1186/s13059-017-1382-0) provided a scalable analysis toolkit, and **best practices** were formalised by Luecken & Theis (*Mol Syst Biol* 2019, doi:10.15252/msb.20188746).

> **Let's trace through this step by step.** Before you run the cell below, read each line and predict what it does. Then run it and check — did you predict correctly?

In [ ]:
# ============================================================
# Section 2 — Simulated Count Matrix
# 500 cells x 2000 genes, 5 cell types
# Negative binomial counts (realistic sparsity)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

# Reproducibility
rng = np.random.default_rng(seed=42)

# ── Parameters ──────────────────────────────────────────────
N_CELLS   = 500
N_GENES   = 2000
N_MITO    = 80   # last 80 genes are mitochondrial (gene names MT-*)
CELL_TYPES = ['T-cell', 'B-cell', 'NK', 'Monocyte', 'Dendritic']
N_TYPES   = len(CELL_TYPES)

# Cells per type (unbalanced, as in real data)
cells_per_type = [150, 120, 80, 100, 50]  # sums to 500
assert sum(cells_per_type) == N_CELLS

# ── Gene name lists ──────────────────────────────────────────
nuclear_genes = [f'GENE{i:04d}' for i in range(1, N_GENES - N_MITO + 1)]
mito_genes    = [f'MT-GENE{i:02d}' for i in range(1, N_MITO + 1)]
all_genes     = nuclear_genes + mito_genes

print(f'Gene list: {len(nuclear_genes)} nuclear + {len(mito_genes)} mitochondrial = {len(all_genes)} total')

# ── Cell-type-specific mean expression profiles ─────────────
# Each cell type has a base expression level per gene drawn from
# an exponential distribution — most genes are lowly expressed.
# A subset of 50 genes per cell type are 'marker genes' with 5x uplift.

base_means = rng.exponential(scale=1.5, size=(N_TYPES, N_GENES))

# Assign marker genes: 50 unique markers per cell type
marker_gene_idx = {}
available = list(range(N_GENES - N_MITO))  # nuclear genes only
for ct_idx, ct in enumerate(CELL_TYPES):
    markers = rng.choice(available, size=50, replace=False)
    marker_gene_idx[ct] = markers.tolist()
    base_means[ct_idx, markers] *= 6.0   # strong marker uplift

# Mitochondrial genes: slightly elevated in monocytes (realistic)
mito_start = N_GENES - N_MITO
base_means[3, mito_start:] *= 2.5  # Monocyte index = 3

# ── Generate count matrix via Negative Binomial ─────────────
# NB(r, p) models overdispersed counts seen in real scRNA-seq data.
# We parameterise via mean mu and dispersion phi: Var = mu + mu^2/phi.
# phi = 1.0 is typical (high overdispersion).

PHI = 1.0  # dispersion parameter

counts = np.zeros((N_CELLS, N_GENES), dtype=np.int32)
cell_type_labels = []

cell_idx = 0
for ct_idx, (ct, n_cells) in enumerate(zip(CELL_TYPES, cells_per_type)):
    mu = base_means[ct_idx]  # shape (N_GENES,)
    # Cell-to-cell library size variation (capture efficiency)
    lib_scale = rng.lognormal(mean=0.0, sigma=0.4, size=n_cells)  # shape (n_cells,)
    
    for j in range(n_cells):
        cell_mu = mu * lib_scale[j]
        # Convert NB parameterisation: p = phi/(mu+phi), r = phi
        p = PHI / (cell_mu + PHI)
        counts[cell_idx] = rng.negative_binomial(n=PHI, p=p)
        cell_type_labels.append(ct)
        cell_idx += 1

cell_type_labels = np.array(cell_type_labels)

# Build DataFrame
cell_names = [f'CELL_{i:04d}' for i in range(N_CELLS)]
count_df = pd.DataFrame(counts, index=cell_names, columns=all_genes)

print(f'\nCount matrix shape: {count_df.shape}  (cells x genes)')
print(f'Sparsity: {(count_df == 0).sum().sum() / count_df.size:.1%} zero entries')
print(f'Median library size: {count_df.sum(axis=1).median():.0f} UMIs/cell')
print(f'\nCell type distribution:')
for ct, n in zip(CELL_TYPES, cells_per_type):
    print(f'  {ct:<12}: {n} cells')

> ⚠️ **Common Mistake:** Running PCA on raw counts instead of normalized data
>
> **Wrong:** `pca = PCA(n_components=2).fit_transform(raw_counts)  # PC1 = library size!`
> **Right:** `normalized = np.log1p(raw_counts / raw_counts.sum(axis=1, keepdims=True) * 1e4)
pca = PCA(n_components=2).fit_transform(normalized)`
> **Why:** Raw counts are dominated by library size (total reads per cell), so PC1 just separates deeply-sequenced cells from shallow ones instead of revealing biology.

> **Expected output:** Gene list counts (nuclear + mitochondrial), count matrix shape, and sparsity percentage  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

In [ ]:
# ============================================================
# Section 3 — Quality Control
# Filter cells by: min_genes detected, max_genes (doublet proxy),
# and mitochondrial gene % (dead/damaged cell proxy)
# ============================================================

# ── Compute per-cell QC metrics ───────────────────────────────
library_size = count_df.sum(axis=1)          # total UMI per cell
n_genes_per_cell = (count_df > 0).sum(axis=1)  # genes detected

mito_cols = [g for g in count_df.columns if g.startswith('MT-')]
mito_counts = count_df[mito_cols].sum(axis=1)
pct_mito = (mito_counts / library_size) * 100

# Data loading/transformation
qc_df = pd.DataFrame({
    'library_size'   : library_size,
    'n_genes'        : n_genes_per_cell,
    'pct_mito'       : pct_mito,
    'cell_type'      : cell_type_labels
})

print('=== Pre-filter QC summary ===')
print(qc_df[['library_size', 'n_genes', 'pct_mito']].describe().round(1))

# ── QC thresholds (literature-informed) ──────────────────────
# Luecken & Theis 2019 recommend:
#   min_genes  > 200 (filters empty droplets)
#   max_genes  < 5000 (filters probable doublets)
#   pct_mito   < 20% (filters dead/damaged cells; tissue-dependent)
MIN_GENES   = 100   # relaxed for simulated data
MAX_GENES   = 1800  # simulated data is 2000 genes wide
MAX_PCT_MITO = 25.0

pass_filter = (
    (qc_df['n_genes'] >= MIN_GENES) &
    (qc_df['n_genes'] <= MAX_GENES) &
    (qc_df['pct_mito'] <= MAX_PCT_MITO)
)

print(f'\nCells passing QC: {pass_filter.sum()} / {N_CELLS} '
      f'({pass_filter.mean():.1%})')
print(f'  Failed min_genes ({MIN_GENES}):  {(qc_df["n_genes"] < MIN_GENES).sum()}')
print(f'  Failed max_genes ({MAX_GENES}):  {(qc_df["n_genes"] > MAX_GENES).sum()}')
print(f'  Failed pct_mito  ({MAX_PCT_MITO}%): {(qc_df["pct_mito"] > MAX_PCT_MITO).sum()}')

# ── QC visualisation ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# Visualization
fig.suptitle('Per-Cell QC Metrics (pre-filter)', fontsize=14, fontweight='bold')

# Build ct_colors mapping
ct_colors = {'T-cell':'#e74c3c', 'B-cell':'#3498db', 'NK':'#2ecc71',
             'Monocyte':'#f39c12', 'Dendritic':'#9b59b6'}

for ct in CELL_TYPES:
    mask = qc_df['cell_type'] == ct
    axes[0].hist(qc_df.loc[mask, 'library_size'], bins=30,
                 alpha=0.6, label=ct, color=ct_colors[ct])
    axes[1].hist(qc_df.loc[mask, 'n_genes'], bins=30,
                 alpha=0.6, label=ct, color=ct_colors[ct])
    axes[2].hist(qc_df.loc[mask, 'pct_mito'], bins=30,
                 alpha=0.6, label=ct, color=ct_colors[ct])

axes[0].set_xlabel('Library size (total UMI)'); axes[0].set_ylabel('Cell count')
axes[0].set_title('Library Size Distribution')
axes[1].set_xlabel('Number of genes detected'); axes[1].set_title('Genes per Cell')
axes[1].axvline(MIN_GENES, color='k', linestyle='--', linewidth=1.5, label=f'min={MIN_GENES}')
axes[1].axvline(MAX_GENES, color='r', linestyle='--', linewidth=1.5, label=f'max={MAX_GENES}')
axes[2].set_xlabel('% mitochondrial reads'); axes[2].set_title('Mitochondrial %')
axes[2].axvline(MAX_PCT_MITO, color='r', linestyle='--', linewidth=1.5, label=f'max={MAX_PCT_MITO}%')

for ax in axes:
    # Visualization
    ax.legend(fontsize=7)

# Visualization
plt.tight_layout()
# Visualization
plt.show()

# Apply filter
count_filt = count_df.loc[pass_filter].copy()
labels_filt = cell_type_labels[pass_filter.values]
qc_filt     = qc_df.loc[pass_filter].copy()
print(f'\nFiltered matrix shape: {count_filt.shape}')

### Why These QC Filters?

| Filter | Removes | Biological rationale |
|--------|---------|---------------------|
| `min_genes < 200` | Empty droplets / ambient RNA | A real cell expresses thousands of genes; near-zero counts = no cell captured |
| `max_genes > 5000` | Doublets | Two cells in one droplet appear to express twice as many genes |
| `pct_mito > 20%` | Dying/damaged cells | Cytoplasmic mRNA leaks out when cell membrane ruptures; mitochondrial mRNA (enclosed in organelle) remains — resulting in artificially high mito % |

In [ ]:
# ============================================================
# Section 4 — Normalisation
# Step 1: Library-size normalisation (counts per 10k, CP10K)
# Step 2: log1p transform
# ============================================================

# ── Why library-size normalisation? ─────────────────────────
# Cell A captured at high efficiency: 8000 UMIs total
# Cell B captured at low efficiency:  2000 UMIs total
# Gene X: 80 counts in A, 20 counts in B  → raw ratio = 4x
# But relative fraction: 80/8000 = 1% in A, 20/2000 = 1% in B
# After CP10K: A → 1000, B → 1000  — same expression, as expected

lib_size = count_filt.sum(axis=1)  # per-cell total UMI

# CP10K: divide each cell by its library size, multiply by 10,000
norm_df = count_filt.div(lib_size, axis=0) * 1e4

# ── Why log1p? ───────────────────────────────────────────────
# After CP10K, expression values are highly right-skewed:
# most genes: 0-100 CP10K, a few housekeeping genes: 5000+ CP10K
# log1p(x) = log(1 + x) compresses this range and makes the
# distribution more Gaussian — required for PCA and distance metrics.
# +1 ensures log(0) is not -inf → log(1+0) = 0 (absent gene stays 0)

log_df = np.log1p(norm_df)

# ── Visualise before/after ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Normalisation Steps', fontsize=13, fontweight='bold')

# Pick a gene with non-trivial expression
gene_example = nuclear_genes[marker_gene_idx['T-cell'][0]]

axes[0].hist(count_filt[gene_example], bins=40, color='steelblue', edgecolor='white', linewidth=0.3)  # transition probability (complement of 0.7)
axes[0].set_title(f'Raw counts: {gene_example}'); axes[0].set_xlabel('UMI count')

axes[1].hist(norm_df[gene_example], bins=40, color='orange', edgecolor='white', linewidth=0.3)  # transition probability (complement of 0.7)
axes[1].set_title(f'After CP10K normalisation'); axes[1].set_xlabel('CP10K')

axes[2].hist(log_df[gene_example], bins=40, color='seagreen', edgecolor='white', linewidth=0.3)  # transition probability (complement of 0.7)
axes[2].set_title(f'After log1p'); axes[2].set_xlabel('log1p(CP10K)')

for ax in axes:
    ax.set_ylabel('Cell count')

plt.tight_layout()
plt.show()

print('Normalisation summary:')
print(f'  Raw       — max: {count_filt[gene_example].max():.0f}, '
      f'median: {count_filt[gene_example].median():.1f}')
print(f'  CP10K     — max: {norm_df[gene_example].max():.1f}, '
      f'median: {norm_df[gene_example].median():.1f}')
print(f'  log1p     — max: {log_df[gene_example].max():.3f}, '
      f'median: {log_df[gene_example].median():.3f}')

### Normalisation Rationale (Academic Detail)

**Library-size normalisation** removes the technical confound of varying capture efficiency between cells. The CP10K (counts per 10,000) convention multiplies by 10,000 so values are on a biologically interpretable scale ("gene X accounts for 1% of transcriptome"). Some pipelines use median library size as the scale factor (Seurat's default), which reduces the range further.

**log1p transform** serves three purposes:
1. **Variance stabilisation:** In count data, variance scales with mean (Poisson: Var = mean; NB: Var = mean + mean²/φ). After log transform, variance is approximately independent of mean — required for PCA, which implicitly assumes equal variance across features.
2. **Dynamic range compression:** Reduces the influence of very highly expressed genes (ribosomal proteins, β-actin) that would otherwise dominate PCA.
3. **Approximate normality:** Many downstream statistical tests assume Gaussian-distributed data.

Note: the +1 pseudocount ensures `log(0)` = 0, preserving sparsity structure.

In [ ]:
# ============================================================
# Section 5 — Highly Variable Gene (HVG) Selection
# Dispersion method: select genes with high mean-normalised variance
# Reference: Satija et al. 2015 (Seurat), Brennecke et al. 2013
# ============================================================

N_HVG = 500  # number of HVGs to select

# ── Compute per-gene mean and variance ──────────────────────
gene_mean = log_df.mean(axis=0)      # mean across cells
gene_var  = log_df.var(axis=0)       # variance across cells

# ── Dispersion: coefficient of variation (variance / mean) ──
# We add a small epsilon to avoid division by zero for silent genes
EPSILON = 1e-8
dispersion = gene_var / (gene_mean + EPSILON)

# ── Bin genes by mean expression, normalise dispersion within bin ─
# This prevents highly expressed genes from dominating solely
# because high expression leads to high absolute variance.
N_BINS = 20
mean_bins = pd.cut(gene_mean, bins=N_BINS, labels=False)
norm_dispersion = dispersion.copy()

for bin_id in range(N_BINS):
    bin_mask = mean_bins == bin_id
    if bin_mask.sum() < 3:
        continue
    bin_disp  = dispersion[bin_mask]
    bin_mean_d = bin_disp.mean()
    bin_std_d  = bin_disp.std() + EPSILON
    norm_dispersion[bin_mask] = (bin_disp - bin_mean_d) / bin_std_d

# Select top N_HVG genes by normalised dispersion
hvg_genes = norm_dispersion.nlargest(N_HVG).index.tolist()

# Remove mitochondrial genes from HVG list (technical, not biological)
hvg_genes = [g for g in hvg_genes if not g.startswith('MT-')]
print(f'HVGs selected (post mito removal): {len(hvg_genes)}')

# Marker gene recovery check
n_markers_recovered = sum(
    nuclear_genes[idx] in hvg_genes
    for ct in CELL_TYPES
    for idx in marker_gene_idx[ct]
)
total_unique_markers = len(set(
    nuclear_genes[idx] for ct in CELL_TYPES for idx in marker_gene_idx[ct]
))
print(f'Marker genes recovered in HVG: {n_markers_recovered}/{total_unique_markers} '
      f'({n_markers_recovered/total_unique_markers:.1%})')

# ── Mean-variance plot ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Highly Variable Gene Selection', fontsize=13, fontweight='bold')

hvg_mask = gene_mean.index.isin(hvg_genes)
axes[0].scatter(gene_mean[~hvg_mask], gene_var[~hvg_mask],
                s=4, alpha=0.3, color='grey', label='Other genes')  # transition probability (complement of 0.7)
axes[0].scatter(gene_mean[hvg_mask], gene_var[hvg_mask],
                s=6, alpha=0.7, color='#e74c3c', label=f'HVG (n={len(hvg_genes)})')  # transition probability (must sum to 1.0 with complement)
axes[0].set_xlabel('Mean log1p expression'); axes[0].set_ylabel('Variance')
axes[0].set_title('Mean–Variance Relationship')
axes[0].legend()

axes[1].scatter(gene_mean[~hvg_mask], norm_dispersion[~hvg_mask],
                s=4, alpha=0.3, color='grey', label='Other')  # transition probability (complement of 0.7)
axes[1].scatter(gene_mean[hvg_mask], norm_dispersion[hvg_mask],
                s=6, alpha=0.7, color='#e74c3c', label='HVG')  # transition probability (must sum to 1.0 with complement)
axes[1].set_xlabel('Mean log1p expression'); axes[1].set_ylabel('Normalised dispersion')
axes[1].set_title('Dispersion Score (binned)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Subset to HVG matrix
hvg_df = log_df[hvg_genes]
print(f'HVG matrix: {hvg_df.shape}')

In [ ]:
# ============================================================
# Section 6 — PCA
# Run on HVG matrix, plot explained variance, PC1 vs PC2
# ============================================================

# Scale to unit variance before PCA (standard practice)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(hvg_df.values)  # shape: (n_cells, n_hvg)

N_PCS = 50
pca = PCA(n_components=N_PCS, random_state=42)
X_pca = pca.fit_transform(X_scaled)  # shape: (n_cells, 50)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print(f'Variance explained by first 10 PCs: {cumulative_var[9]:.1%}')
print(f'PCs needed to reach 80% variance:   {np.searchsorted(cumulative_var, 0.80) + 1}')

# ── Plots ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 5))
gs = gridspec.GridSpec(1, 3, figure=fig)
fig.suptitle('PCA on HVG Matrix', fontsize=13, fontweight='bold')

# Scree plot
ax0 = fig.add_subplot(gs[0])
ax0.bar(range(1, 21), explained_var[:20] * 100, color='steelblue', alpha=0.8)
ax0.set_xlabel('Principal Component'); ax0.set_ylabel('% Variance Explained')
ax0.set_title('Scree Plot (first 20 PCs)')

# Cumulative variance
ax1 = fig.add_subplot(gs[1])
ax1.plot(range(1, N_PCS + 1), cumulative_var * 100, 'o-', color='steelblue', markersize=3)
ax1.axhline(80, color='red', linestyle='--', label='80%')
ax1.set_xlabel('Number of PCs'); ax1.set_ylabel('Cumulative % Variance')
ax1.set_title('Elbow Plot'); ax1.legend()

# PC1 vs PC2 coloured by cell type
ax2 = fig.add_subplot(gs[2])
for ct in CELL_TYPES:
    mask = labels_filt == ct
    # Use filtered labels aligned to filtered count matrix
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1],
                s=18, alpha=0.75, label=ct, color=ct_colors[ct])
ax2.set_xlabel(f'PC1 ({explained_var[0]:.1%} var)')
ax2.set_ylabel(f'PC2 ({explained_var[1]:.1%} var)')
ax2.set_title('PC1 vs PC2 by Cell Type')
ax2.legend(markerscale=1.5, fontsize=8)

plt.tight_layout()
plt.show()

### Reading a PCA Plot

**Scree plot:** Each bar is the variance explained by one PC. A sharp 'elbow' indicates how many PCs capture real biological signal before the rest is noise. For scRNA-seq, typically 10–50 PCs are biologically meaningful.

**PC1 vs PC2 scatter:** If cell types cluster cleanly in 2D PCA, the first two PCs capture most of the cell-type variation. In real data this rarely holds — you typically need 10–30 PCs before passing to UMAP or t-SNE. Here the separation is clear because our simulation has strong marker genes.

**Why scale before PCA?** PCA is variance-maximising. Without scaling, genes with large raw variance (housekeeping genes) dominate the first PCs, masking biologically relevant variation in lowly expressed genes.

In [ ]:
# ============================================================
# Section 7 — t-SNE Visualisation
# Used here as a proxy for UMAP (umap-learn not guaranteed).
# Both are non-linear dimensionality reduction methods that
# preserve local neighbourhood structure in 2D.
# ============================================================

# Standard practice: run t-SNE on top 20 PCs, not on full HVG matrix
# This denoises the input and speeds up computation.
N_PCS_FOR_TSNE = 20
X_pca_sub = X_pca[:, :N_PCS_FOR_TSNE]

print('Running t-SNE (may take ~30s)...')
tsne = TSNE(
    n_components=2,
    perplexity=30,       # controls effective neighbourhood size (~5-50, default 30)
    learning_rate='auto',
    n_iter=1000,
    random_state=42,
    init='pca'           # PCA initialisation reduces random noise in final layout
)
X_tsne = tsne.fit_transform(X_pca_sub)
print(f't-SNE embedding shape: {X_tsne.shape}')

# ── Plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('t-SNE Embedding (proxy for UMAP)', fontsize=13, fontweight='bold')

# Colour by cell type
for ct in CELL_TYPES:
    mask = labels_filt == ct
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    s=20, alpha=0.8, label=ct, color=ct_colors[ct])
axes[0].set_title('Coloured by True Cell Type')
axes[0].set_xlabel('t-SNE 1'); axes[0].set_ylabel('t-SNE 2')
axes[0].legend(markerscale=1.5, fontsize=9)

# Colour by library size (technical variable — should not drive structure)
sc = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                     c=np.log1p(qc_filt['library_size'].values),
                     s=20, alpha=0.7, cmap='viridis')  # transition probability (must sum to 1.0 with complement)
plt.colorbar(sc, ax=axes[1], label='log1p(library size)')
axes[1].set_title('Coloured by Library Size (QC check)\nShould NOT drive cluster structure')
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')

plt.tight_layout()
plt.show()

print('\nt-SNE parameters note:')
print('  perplexity=30: roughly, the effective number of local neighbours.')
print('  Higher perplexity → more global structure visible.')
print('  t-SNE preserves LOCAL distances; inter-cluster distances are NOT meaningful.')
print('  UMAP additionally preserves more global structure (McInnes 2018 arxiv:1802.03426)')

## 🏁 Checkpoint -- Pause and Verify

Before continuing, make sure you can:
1. Explain why scRNA-seq data has dropouts and how normalization handles them
2. Run PCA on a counts matrix and interpret the first two principal components
3. Describe the QC filtering steps (min genes per cell, min cells per gene, mito %) and why each matters

**If any of these feel shaky**, re-read the section above or review the prerequisite notebook listed in the Cross-References section.

### t-SNE vs UMAP — Key Differences

| Property | t-SNE | UMAP |
|----------|-------|------|
| **Global structure** | Poor — inter-cluster distances not meaningful | Better preserved |
| **Speed** | O(N log N) with BH approximation | Faster, scales to millions of cells |
| **Determinism** | Stochastic (random_state helps) | More stable |
| **Theoretical basis** | KL divergence on probability distributions | Riemannian geometry + topological data analysis |
| **Key parameter** | `perplexity` | `n_neighbors`, `min_dist` |

**Critical caveat:** Neither t-SNE nor UMAP should be used for quantitative comparisons. Two clusters appearing close in t-SNE does *not* mean they are transcriptomically similar. Always use high-dimensional distances (PCA space) for statistical comparisons.

Reference: McInnes, L., Healy, J., & Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction. *arXiv:1802.03426*.

In [ ]:
# ============================================================
# Section 8 — Leiden-style Clustering
# We implement: KNN graph construction + KMeans as a
# computationally accessible proxy for Leiden/Louvain
# (which require the leidenalg/igraph packages).
# For production: use scanpy.tl.leiden() or igraph.
# ============================================================

# ── Step 1: Build KNN graph in PCA space ────────────────────
K_NEIGHBORS = 15  # typical for scRNA-seq (10–30)
knn_graph = kneighbors_graph(
    X_pca_sub, n_neighbors=K_NEIGHBORS,
    mode='connectivity', include_self=False
)
# Symmetrise: A = (A + A^T) / 2  (undirected graph)
adj_matrix = (knn_graph + knn_graph.T)
adj_matrix.data = np.ones_like(adj_matrix.data)  # binary

print(f'KNN graph: {adj_matrix.shape}, '
      f'{adj_matrix.nnz} edges (mean degree = {adj_matrix.nnz/N_CELLS:.1f})')

# ── Step 2: KMeans clustering (proxy for Leiden) ─────────────
# In real Leiden: communities are found by maximising modularity
# on the KNN graph. KMeans in PCA space is a reasonable proxy.
N_CLUSTERS = 5  # we know the true number from simulation
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_pca_sub)

# ── Step 3: Compare clusters to true cell types ─────────────
# Confusion matrix approach: for each cluster, find the dominant cell type
cluster_to_type = {}
for cl in range(N_CLUSTERS):
    mask = cluster_labels == cl
    ct_counts = pd.Series(labels_filt[mask]).value_counts()
    dominant = ct_counts.index[0]
    purity = ct_counts.iloc[0] / mask.sum()
    cluster_to_type[cl] = dominant
    print(f'Cluster {cl} → {dominant:<12} (purity: {purity:.1%}, n={mask.sum()})')

# Assign predicted labels
predicted_types = np.array([cluster_to_type[cl] for cl in cluster_labels])
accuracy = (predicted_types == labels_filt).mean()
print(f'\nOverall clustering accuracy (vs true labels): {accuracy:.1%}')

# ── Visualise in t-SNE space ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Clustering Results', fontsize=13, fontweight='bold')

cluster_palette = plt.cm.tab10(np.linspace(0, 1, N_CLUSTERS))

for cl in range(N_CLUSTERS):
    mask = cluster_labels == cl
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    s=20, alpha=0.7,  # transition probability (must sum to 1.0 with complement)
                    label=f'Cluster {cl} ({cluster_to_type[cl]})',
                    color=cluster_palette[cl])
axes[0].set_title(f'KMeans Clusters (k={N_CLUSTERS})')
axes[0].set_xlabel('t-SNE 1'); axes[0].set_ylabel('t-SNE 2')
axes[0].legend(fontsize=8, markerscale=1.5)

for ct in CELL_TYPES:
    mask = labels_filt == ct
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    s=20, alpha=0.7, label=ct, color=ct_colors[ct])  # transition probability (must sum to 1.0 with complement)
axes[1].set_title('True Cell Types (ground truth)')
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')
axes[1].legend(fontsize=9, markerscale=1.5)

plt.tight_layout()
plt.show()

# Confusion matrix
print('\nConfusion matrix (rows=true, cols=cluster):')
conf_df = pd.crosstab(
    pd.Series(labels_filt, name='True'),
    pd.Series(cluster_labels, name='Cluster')
)
print(conf_df)

> **Why this code:** This performs differential expression analysis to identify marker genes that distinguish each cell cluster.

In [ ]:
# ============================================================
# Section 9 — Differential Expression & Marker Gene Discovery
# For each cluster vs all other cells:
#   1. Compute log2 fold-change (log2FC)
#   2. Mann-Whitney U test (non-parametric, appropriate for sparse counts)
#   3. Report top 5 marker genes per cluster
# Reference: Luecken & Theis 2019 recommend Wilcoxon/MWU over t-test
# for scRNA-seq (non-normal distribution, many zeros)
# ============================================================

from scipy.stats import mannwhitneyu

# Use log-normalised data (log_df), subset to filtered cells
log_arr = log_df.values   # shape (n_cells_filt, n_genes)
gene_names = log_df.columns.tolist()

def find_markers(cluster_id, cluster_labels, expression_matrix, gene_list,
                 n_top=5, min_pct=0.10):
    """
    Find marker genes for one cluster vs all others.
    
    Parameters
    ----------
    cluster_id      : int, the cluster to test
    cluster_labels  : array of cluster IDs per cell
    expression_matrix: (n_cells, n_genes) log-normalised expression
    gene_list       : list of gene names
    n_top           : number of top markers to return
    min_pct         : minimum fraction of cluster cells expressing gene (>0)
    
    Returns
    -------
    DataFrame with columns: gene, log2FC, pvalue, pct_in, pct_out
    """
    in_mask  = cluster_labels == cluster_id
    out_mask = ~in_mask
    
    X_in  = expression_matrix[in_mask]   # (n_in, n_genes)
    X_out = expression_matrix[out_mask]  # (n_out, n_genes)
    
    results = []
    for g_idx, gene in enumerate(gene_list):
        vals_in  = X_in[:, g_idx]
        vals_out = X_out[:, g_idx]
        
        pct_in  = (vals_in  > 0).mean()
        pct_out = (vals_out > 0).mean()
        
        # Skip genes not expressed in enough cells
        if pct_in < min_pct:
            continue
        
        # log2FC: mean log expression in vs out
        mean_in  = vals_in.mean()
        mean_out = vals_out.mean()
        log2fc   = mean_in - mean_out  # log space difference = log ratio
        
        # Mann-Whitney U test (one-sided: in > out)
        try:
            _, pval = mannwhitneyu(vals_in, vals_out, alternative='greater')
        except ValueError:
            pval = 1.0
        
        results.append({'gene': gene, 'log2FC': log2fc,
                        'pvalue': pval, 'pct_in': pct_in, 'pct_out': pct_out})
    
    if not results:
        return pd.DataFrame()
    
    res_df = pd.DataFrame(results)
    
    # Bonferroni correction (conservative)
    res_df['pval_adj'] = np.minimum(res_df['pvalue'] * len(results), 1.0)
    
    # Sort by log2FC (upregulated markers)
    res_df = res_df.sort_values('log2FC', ascending=False)
    return res_df.head(n_top)

# ── Run DE for each cluster ────────────────────────────────
print('Differential expression — top 5 markers per cluster:')
print('=' * 70)

all_markers = {}
for cl in range(N_CLUSTERS):
    markers = find_markers(
        cl, cluster_labels, log_arr, gene_names,
        n_top=5, min_pct=0.10
    )
    all_markers[cl] = markers
    print(f'\nCluster {cl} ({cluster_to_type[cl]}):')
    if markers.empty:
        print('  No markers found.')
    else:
        print(markers[['gene', 'log2FC', 'pval_adj', 'pct_in', 'pct_out']].to_string(index=False))

In [ ]:
# ── Dot plot of marker genes ──────────────────────────────────
# Shows: dot size = fraction of cells expressing gene (pct_in)
#        dot colour = mean expression in cluster

top1_per_cluster = []
# --- Loop over range(N_CLUSTERS) ---
for cl in range(N_CLUSTERS):
    m = all_markers[cl]
    if not m.empty:
        top1_per_cluster.append(m.iloc[0]['gene'])

# Collect top 3 markers per cluster for dot plot
plot_genes = []
plot_labels_cluster = []
# --- Loop over range(N_CLUSTERS) ---
for cl in range(N_CLUSTERS):
    m = all_markers[cl]
    if not m.empty:
        # --- Loop ---
        for _, row in m.head(3).iterrows():
            plot_genes.append(row['gene'])
            plot_labels_cluster.append(f'C{cl}({cluster_to_type[cl][:3]})')

# Unique genes
unique_genes = list(dict.fromkeys(plot_genes))

if unique_genes:
    # Build matrix: clusters x genes
    mean_expr = np.zeros((N_CLUSTERS, len(unique_genes)))
    pct_expr  = np.zeros((N_CLUSTERS, len(unique_genes)))
    
    # --- Loop over range(N_CLUSTERS) ---
    for cl in range(N_CLUSTERS):
        mask = cluster_labels == cl
        # --- Loop ---
        for gi, gene in enumerate(unique_genes):
            if gene in log_df.columns:
                vals = log_df[gene].values[mask]
                mean_expr[cl, gi] = vals.mean()
                pct_expr[cl, gi]  = (vals > 0).mean()
    
    fig, ax = plt.subplots(figsize=(max(10, len(unique_genes) * 0.6), 4))
    
    cluster_names_short = [f'C{cl}\n({cluster_to_type[cl][:4]})' for cl in range(N_CLUSTERS)]
    vmin, vmax = mean_expr.min(), mean_expr.max()
    
    # --- Loop over range(N_CLUSTERS) ---
    for ci in range(N_CLUSTERS):
        # --- Loop over range(len(unique_genes)) ---
        for gi in range(len(unique_genes)):
            size = pct_expr[ci, gi] * 300
            color_val = (mean_expr[ci, gi] - vmin) / (vmax - vmin + 1e-8)
            ax.scatter(gi, ci, s=size, c=[plt.cm.Reds(color_val)],
                      edgecolors='grey', linewidths=0.3)  # transition probability (complement of 0.7)
    
    ax.set_xticks(range(len(unique_genes)))
    ax.set_xticklabels(unique_genes, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(N_CLUSTERS))
    ax.set_yticklabels(cluster_names_short, fontsize=8)
    ax.set_title('Marker Gene Dot Plot\n(size = % expressing, colour = mean expression)',
                 fontsize=11)
    ax.grid(True, alpha=0.2)
    
    # Legend
    for size_val, label in [(0.1*300, '10%'), (0.5*300, '50%'), (1.0*300, '100%')]:
        ax.scatter([], [], s=size_val, c='grey', label=label)
    ax.legend(title='% expressing', loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()
else:
    print('No marker genes to plot.')

> ✋ **Try it yourself:** Before running the next cell, predict what the output will be. Then run it and check. If you were wrong, re-read the code above.

In [ ]:
# ============================================================
# Section 10 — Pseudotime / Trajectory Analysis
# T-cell activation: naive T-cell → effector T-cell
# Method: project T-cells onto their first PCA dimension,
# use this as a 1D pseudotime axis.
# Reference: Trapnell et al. 2014 (Monocle) doi:10.1038/nbt.2859
# ============================================================

# ── Select T-cells only ────────────────────────────────────
tcell_mask = labels_filt == 'T-cell'
n_tcells = tcell_mask.sum()
print(f'T-cells in filtered dataset: {n_tcells}')

# ── Simulate activation gradient ───────────────────────────
# We assign a latent 'activation score' to each T-cell
# (in real data this emerges from the data itself)
# Activation score: uniform from 0 (naive) to 1 (effector)
tcell_idx_global = np.where(tcell_mask)[0]
activation_score = rng.uniform(0, 1, size=n_tcells)
activation_score = np.sort(activation_score)  # ordered for simulation realism

# ── Simulate activation-dependent gene expression ──────────
# Genes upregulated in effector T-cells: GZMB, IFNG, PRF1-like
# Genes upregulated in naive T-cells: TCF7, LEF1, CCR7-like
effector_genes_idx = marker_gene_idx['T-cell'][:10]    # first 10 T-cell markers
naive_genes_idx    = marker_gene_idx['T-cell'][10:20]   # next 10

# Build a T-cell expression sub-matrix
tcell_expr = log_df.values[tcell_mask]  # (n_tcells, n_genes)

# Add activation gradient to effector and naive genes
for gi in effector_genes_idx:
    gene_name = nuclear_genes[gi]
    if gene_name in log_df.columns:
        col_idx = log_df.columns.get_loc(gene_name)
        # Positively correlated with activation
        tcell_expr[:, col_idx] += activation_score * 2.0 + rng.normal(0, 0.3, n_tcells)  # transition probability (complement of 0.7)

for gi in naive_genes_idx:
    gene_name = nuclear_genes[gi]
    if gene_name in log_df.columns:
        col_idx = log_df.columns.get_loc(gene_name)
        # Negatively correlated with activation
        tcell_expr[:, col_idx] += (1.0 - activation_score) * 2.0 + rng.normal(0, 0.3, n_tcells)  # transition probability (complement of 0.7)

# ── Run PCA on T-cells only to derive pseudotime ────────────
tcell_scaler = StandardScaler()
tcell_scaled = tcell_scaler.fit_transform(tcell_expr)
tcell_pca = PCA(n_components=5, random_state=42)
tcell_pca_coords = tcell_pca.fit_transform(tcell_scaled)
pseudotime = tcell_pca_coords[:, 0]  # PC1 as pseudotime

# Correlate PC1 with true activation score to get direction
from scipy.stats import pearsonr
r, _ = pearsonr(pseudotime, activation_score)
if r < 0:  # flip if anti-correlated
    pseudotime = -pseudotime

# Normalise pseudotime to [0, 1]
pseudotime = (pseudotime - pseudotime.min()) / (pseudotime.max() - pseudotime.min())
print(f'Correlation of PC1 pseudotime with true activation: |r| = {abs(r):.3f}')

# ── Plot trajectories ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('T-cell Activation Trajectory (Pseudotime)', fontsize=13, fontweight='bold')

# Panel 1: Pseudotime vs true activation
axes[0].scatter(pseudotime, activation_score, s=25, alpha=0.7,  # transition probability (must sum to 1.0 with complement)
                c=activation_score, cmap='RdYlBu_r')
axes[0].set_xlabel('PC1 Pseudotime (inferred)'); axes[0].set_ylabel('True Activation Score')
axes[0].set_title(f'Pseudotime vs Ground Truth\n|r| = {abs(r):.3f}')

# Panel 2: Effector gene expression along pseudotime
eg_name = nuclear_genes[effector_genes_idx[0]] if nuclear_genes[effector_genes_idx[0]] in log_df.columns else None
if eg_name:
    col_idx = log_df.columns.get_loc(eg_name)
    eff_expr = tcell_expr[:, col_idx]
    sort_order = np.argsort(pseudotime)
    # Smooth with rolling mean
    smooth_window = max(5, n_tcells // 15)
    smoothed = pd.Series(eff_expr[sort_order]).rolling(smooth_window, center=True).mean()
    axes[1].scatter(pseudotime[sort_order], eff_expr[sort_order],
                    s=15, alpha=0.4, c='#e74c3c')
    axes[1].plot(pseudotime[sort_order], smoothed, 'r-', linewidth=2,
                 label=f'{eg_name} (effector)')

ng_name = nuclear_genes[naive_genes_idx[0]] if nuclear_genes[naive_genes_idx[0]] in log_df.columns else None
if ng_name:
    col_idx2 = log_df.columns.get_loc(ng_name)
    naive_expr = tcell_expr[:, col_idx2]
    smoothed2 = pd.Series(naive_expr[sort_order]).rolling(smooth_window, center=True).mean()
    axes[1].scatter(pseudotime[sort_order], naive_expr[sort_order],
                    s=15, alpha=0.4, c='#3498db')
    axes[1].plot(pseudotime[sort_order], smoothed2, 'b-', linewidth=2,
                 label=f'{ng_name} (naive)')

axes[1].set_xlabel('Pseudotime'); axes[1].set_ylabel('log1p expression')
axes[1].set_title('Gene Expression Along Pseudotime')
axes[1].legend(fontsize=8)
axes[1].annotate('Naive', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=10,  # standard p-value significance threshold
                 color='#3498db', fontweight='bold')
axes[1].annotate('Effector', xy=(0.7, 0.95), xycoords='axes fraction', fontsize=10,  # transition probability (must sum to 1.0 with complement)
                 color='#e74c3c', fontweight='bold')

# Panel 3: T-cells coloured by pseudotime in t-SNE space
tsne_tcell = X_tsne[tcell_mask]
sc2 = axes[2].scatter(tsne_tcell[:, 0], tsne_tcell[:, 1],
                      c=pseudotime, cmap='RdYlBu_r', s=40, alpha=0.8)  # MMseqs2 clustering identity threshold (%)
plt.colorbar(sc2, ax=axes[2], label='Pseudotime')
axes[2].set_xlabel('t-SNE 1'); axes[2].set_ylabel('t-SNE 2')
axes[2].set_title('T-cells in t-SNE Space\nColoured by Pseudotime')

plt.tight_layout()
plt.show()

### Pseudotime Analysis — Academic Context

**Monocle (Trapnell et al. 2014)** introduced the concept of pseudotime: ordering cells along a biological process (differentiation, activation) using the structure of the high-dimensional expression data. Key insight: a tissue snapshot contains cells at many stages of a dynamic process, so the *ordering* of cells reveals the trajectory.

**Assumptions:**
1. Cells progress through a continuous state space (no sudden jumps)
2. The dominant axis of variation (PC1) corresponds to the biological process of interest
3. Sufficient cells are captured at intermediate stages

**Modern methods:** RNA velocity (La Manno et al. 2018, *Nature*) uses the ratio of spliced:unspliced mRNA to infer the *direction* of differentiation, not just the ordering. This is now the gold standard.

**What makes T-cell activation tractable:**
- Naive T-cells express CCR7, TCF7, LEF1 (transcription factors for quiescence)
- Effector T-cells upregulate GZMB (granzyme B), IFNG (interferon-γ), PRF1 (perforin)
- This is one of the best-characterised differentiation axes in immunology

## Section 11 — Connection to Protein ML and Drug Discovery

### How scRNA-seq Identifies Drug Targets

scRNA-seq transforms drug discovery in three key ways:

**1. Cell-type-specific expression = target identification**  
If a gene of interest is only highly expressed in one cell type (e.g., a kinase expressed exclusively in dendritic cells), a small molecule targeting its protein product will have a narrow therapeutic window. scRNA-seq replaces the need to purify individual cell types before measuring expression.

**Example:** CAR-T cell therapy targets CD19, a surface protein expressed almost exclusively on B-cells and B-cell tumours. scRNA-seq confirmed CD19's restricted expression profile, validating it as a safe target.

**2. Disease-altered expression = mechanistic insight**  
Comparing healthy vs disease single-cell atlases (e.g., the Human Cell Atlas) reveals which cell types and pathways are most perturbed. This points to which proteins undergo conformational change, post-translational modification, or altered complex formation — all targets for structural biology.

**3. Prioritising protein structures for AlphaFold/MD simulation**  
Computing time for MD simulation or experimental cryo-EM is expensive. scRNA-seq data informs which proteins are:
- Highly expressed in disease-relevant cell types → worth structural characterisation
- Differentially expressed between conditions → potential allosteric pockets emerge
- Co-expressed in regulatory networks → protein–protein interaction targets

### Concrete Pipeline

```
scRNA-seq (Module 02/05)
       ↓
Cell-type-specific DE genes
       ↓
Pathway enrichment (Module 02/04)
       ↓
Target protein identified
       ↓
Structure prediction with AlphaFold3 (Module 07)
       ↓
Binding site analysis + MD simulation (Module 11)
       ↓
Drug design / fine-tuning (Module 10)
```

### Cross-Module Links

| This notebook... | Connects to... |
|-----------------|----------------|
| HVG selection (dispersion method) | Module 04/01: PCA + UMAP for gene expression ML |
| Differential expression (MWU test) | Module 02/02: bulk DE, DESeq2, volcano plots |
| Clustering quality metrics | Module 09/01: silhouette score, learning curves |
| Pseudotime → protein expression dynamics | Module 07: AlphaFold3 conformational ensembles |

## Section 12 — Interview Q&A

### Five Core Interview Questions

---

**Q1. What is a UMI and why does it matter?**

A **Unique Molecular Identifier** is a short (6–12 nt) random nucleotide barcode ligated to each cDNA molecule *before* PCR amplification. During sequencing, all reads from the same original molecule share the same UMI. After sequencing, reads with the same cell barcode + gene + UMI are collapsed into a single count, removing PCR duplicates. Without UMIs, PCR amplification bias (some molecules amplify 10x more than others) would make expression measurements unreliable. UMIs allow absolute molecule counting rather than read counting.

---

**Q2. Why must you normalise scRNA-seq data and what are the steps?**

Raw UMI counts vary between cells due to **technical capture efficiency** (how well the bead captured mRNA from a given cell), not biological differences. A cell with 5000 UMIs may appear to express everything at twice the level of a cell with 2500 UMIs, but they may have identical biology. Normalisation steps:

1. **Library-size normalisation (CP10K):** Divide each cell's counts by total UMI count, multiply by 10,000. Now all cells have the same total count, making them comparable.
2. **log1p transform:** Compresses the right-skewed distribution, stabilises variance (required for PCA), and prevents highly expressed housekeeping genes from dominating.

Advanced methods (SCTransform, scran pooling) model the mean-variance relationship explicitly using negative binomial regression.

---

**Q3. What does UMAP preserve, and what should you NOT conclude from a UMAP plot?**

UMAP (McInnes 2018) preserves **local neighbourhood structure**: cells that are transcriptomically similar (nearby in high-dimensional space) tend to appear close in the 2D UMAP embedding. It also preserves *more* global structure than t-SNE, meaning cluster-to-cluster distances are partially meaningful.

**Do NOT conclude:** The distances between clusters are not quantitatively interpretable. Two clusters appearing far apart in UMAP may be only slightly different in the 50-PC space. Always use high-dimensional distances (e.g., PCA cosine distance) for quantitative comparisons. Also, cluster shapes and sizes in UMAP are distorted and should not be interpreted literally.

---

**Q4. How do you find marker genes for a cell cluster?**

Standard approach (Wilcoxon rank-sum / Mann-Whitney U test):
1. For each gene, compare expression in cluster X vs all other cells
2. Compute log2 fold-change (mean in X − mean in rest, in log space)
3. Compute Mann-Whitney U p-value (non-parametric, appropriate for sparse non-normal data)
4. Apply multiple testing correction (Bonferroni or Benjamini-Hochberg FDR)
5. Filter by `pct_in > 0.25` (gene expressed in >25% of cluster) and `log2FC > 0.5`
6. Rank by log2FC or by adjusted p-value

Why not a t-test? scRNA-seq data is highly non-normal (many zeros, heavy right tail). The MWU test makes no distributional assumptions. Luecken & Theis 2019 benchmarked methods and recommend the Wilcoxon test as the default.

---

**Q5. What is the difference between Leiden and Louvain clustering, and why are they preferred over k-means for scRNA-seq?**

Both Leiden (Traag et al. 2019) and Louvain are **graph-based community detection** methods that operate on a KNN graph of cells. They optimise modularity — a measure of cluster density relative to a random graph.

**Leiden vs Louvain:** Louvain can produce internally disconnected communities (a bug where a cluster contains cells not connected to each other). Leiden guarantees well-connected partitions and is now the recommended default.

**Why not k-means?** k-means assumes (1) spherical clusters of equal size and (2) Euclidean distance is meaningful in high dimensions. Single-cell clusters are non-spherical, unequal in size, and the curse of dimensionality makes Euclidean distance unreliable. Graph-based methods avoid all these assumptions by operating on local connectivity, not global geometry.

## Section 13 — Debug Exercise

The following cell contains **three deliberate bugs** in the QC/normalisation pipeline. Find and fix all three before running.

**Hint:** Run the cell as-is first. Some bugs produce wrong results silently (no error raised). Read the output carefully.

**Learning goal:** These are the most common scRNA-seq pipeline mistakes seen in real codebases.

In [ ]:
# ============================================================
# DEBUG EXERCISE — Find and fix 3 bugs
# Run as-is first, then fix each bug
# ============================================================

import numpy as np
import pandas as pd

# Simulated mini dataset: 10 cells x 20 genes
rng_debug = np.random.default_rng(seed=99)
debug_counts = rng_debug.negative_binomial(n=1, p=0.5, size=(10, 20))

# Add 5 'mitochondrial' genes (columns 15-19)
debug_genes = [f'GENE{i}' for i in range(15)] + [f'MT-G{i}' for i in range(5)]
debug_df = pd.DataFrame(debug_counts, columns=debug_genes)

# ── BUG 1: Computing mitochondrial percentage ────────────────
mito_cols_debug = [g for g in debug_df.columns if g.startswith('MT-')]
mito_sum = debug_df[mito_cols_debug].sum(axis=1)
total_sum = debug_df.sum(axis=1)

# BUG: dividing by total number of genes instead of total UMI count
pct_mito_buggy = mito_sum / len(debug_genes) * 100  # <-- BUG 1 HERE
print(f'Mito % (buggy): {pct_mito_buggy.values.round(1)}')
# Should be: mito_sum / total_sum * 100

# ── BUG 2: Library-size normalisation ────────────────────────
lib_sizes = debug_df.sum(axis=1)

# BUG: normalising columns (axis=1) instead of rows (axis=0)
norm_buggy = debug_df.div(lib_sizes, axis=1) * 1e4  # <-- BUG 2 HERE
print(f'\nRow sums after buggy norm (should all equal 10000):')
print(norm_buggy.sum(axis=1).values.round(1))
# Should be: debug_df.div(lib_sizes, axis=0) * 1e4

# ── BUG 3: log1p transform ────────────────────────────────────
# After correct normalisation:
norm_correct = debug_df.div(lib_sizes, axis=0) * 1e4

# BUG: using np.log instead of np.log1p
# np.log(0) = -inf, breaking all downstream analysis
log_buggy = np.log(norm_correct + 0)   # <-- BUG 3 HERE (missing +1 in log1p, or wrong function)
print(f'\n-inf values after log(0) transform: {(log_buggy == -np.inf).sum().sum()}')
# Should be: np.log1p(norm_correct)  OR  np.log(norm_correct + 1)

print('\n--- ANSWERS (spoiler) ---')
print('Bug 1: divide by total_sum (UMI), not len(debug_genes) (gene count)')
print('Bug 2: axis=0 (divide each row by its lib size), not axis=1')
print('Bug 3: np.log1p(x) avoids log(0)=-inf; or use np.log(x + 1)')

In [ ]:
# ── CORRECTED PIPELINE (run after finding bugs) ──────────────

# Bug 1 fix: pct_mito = mito_sum / total_sum * 100
pct_mito_correct = mito_sum / total_sum * 100

# Bug 2 fix: axis=0
norm_correct = debug_df.div(lib_sizes, axis=0) * 1e4

# Bug 3 fix: log1p
log_correct = np.log1p(norm_correct)

print('=== Corrected results ===')
print(f'Mito % (correct): {pct_mito_correct.values.round(1)}')
print(f'Row sums after correct norm (all should be 10000): {norm_correct.sum(axis=1).values.round(0)}')
print(f'-inf values after log1p: {(log_correct == -np.inf).sum().sum()}  (should be 0)')
print(f'Zero entries after log1p: {(log_correct == 0).sum().sum()}  (correctly represents absent genes)')

## Section 14 — Primary Literature & Resources

### Foundational Papers

| Paper | Key Contribution | DOI / arXiv |
|-------|-----------------|-------------|
| Macosko et al. 2015. *Cell* | Drop-seq: first massively parallel scRNA-seq using barcoded beads in droplets; profiled 44,808 mouse retina cells | doi:10.1016/j.cell.2015.05.002 |
| Wolf et al. 2018. *Genome Biology* | Scanpy: scalable Python framework for single-cell analysis; graph-based clustering, PAGA trajectory inference | doi:10.1186/s13059-017-1382-0 |
| Luecken & Theis 2019. *Mol Syst Biol* | Best practices for scRNA-seq analysis: QC, normalisation, HVG, clustering, DE benchmarks | doi:10.15252/msb.20188746 |
| McInnes et al. 2018. *arXiv* | UMAP: Uniform Manifold Approximation and Projection — faster and topology-preserving alternative to t-SNE | arxiv:1802.03426 |
| Trapnell et al. 2014. *Nature Biotechnology* | Monocle: pseudotime ordering of cells along developmental trajectories | doi:10.1038/nbt.2859 |
| Traag et al. 2019. *Scientific Reports* | Leiden algorithm: guaranteed-connected communities in KNN graphs | doi:10.1038/s41598-019-41695-z |
| La Manno et al. 2018. *Nature* | RNA velocity: splicing kinetics reveal differentiation direction | doi:10.1038/s41586-018-0414-6 |
| Zheng et al. 2017. *Nature Communications* | 10x Genomics: commercial droplet scRNA-seq, demonstrated on 68,000 PBMCs | doi:10.1038/ncomms14049 |

### Recommended Tools (Production Pipeline)

| Tool | Language | Purpose |
|------|----------|---------|
| Scanpy | Python | Full scRNA-seq pipeline (this notebook reimplements its core) |
| Seurat v5 | R | Dominant R ecosystem, multimodal integration |
| scVI | Python | Deep generative model for batch correction and DE |
| CellTypist | Python | Automated cell type annotation using logistic regression |
| Velocyto / scVelo | Python | RNA velocity trajectory analysis |
| BBKNN | Python | Batch-balanced KNN for multi-sample integration |
| Harmony | R/Python | Fast linear integration of multiple single-cell datasets |

### Datasets for Practice

| Dataset | Cells | Description | Source |
|---------|-------|-------------|--------|
| 10x PBMC 3k | 2,700 | Human peripheral blood mononuclear cells | 10x Genomics website (free) |
| Human Cell Atlas | 500k+ | Multi-tissue human cell atlas | humancellatlas.org |
| Tabula Muris | 100k | Mouse organ survey (Calico/Chan-Zuckerberg) | cziscience.com |
| COVID-19 Cell Atlas | 1.4M | Cells from COVID-19 patients and controls | covid19cellatlas.org |

## Section 15 — Mastery Check (Bloom's Taxonomy)

Answer these questions in order. Each level builds on the last.

---

### Level 1 — Remember

**Q:** What are the three standard QC filters applied in scRNA-seq and what technical artefact does each remove?

<details>
<summary>Answer</summary>

1. **min_genes > 200** → removes empty droplets (no real cell captured, only ambient RNA)
2. **max_genes < 5000** → removes doublets (two cells in one droplet, appears to express twice as many genes)
3. **pct_mito < 20%** → removes dying/damaged cells (cytoplasmic mRNA leaks out, mitochondrial mRNA stays, inflating mito %)

</details>

---

### Level 2 — Understand

**Q:** Explain why high mitochondrial percentage indicates a dying cell, not a metabolically active one.

<details>
<summary>Answer</summary>

Mitochondria are enclosed organelles with a double lipid bilayer membrane. When a cell's outer (plasma) membrane ruptures during lysis (due to damage or death), cytoplasmic mRNA leaks out of the droplet before it can be captured and sequenced. Mitochondrial mRNA, being physically enclosed in the organelle, is retained. The result: a damaged cell appears to have a disproportionate fraction of mitochondrial transcripts — not because of mitochondrial upregulation, but because nuclear transcripts were lost. In healthy cells, pct_mito is typically 5–15%.

Note: some tissues (cardiomyocytes, skeletal muscle) have legitimately high mito content — thresholds should be set per tissue type.

</details>

---

### Level 3 — Apply

**Q:** You receive a new dataset with 10,000 cells. After QC, you have 4,000 cells. t-SNE shows 8 clusters. You run Wilcoxon DE and find that Cluster 3 has no significant marker genes (all adjusted p-values > 0.05). List three possible biological or technical explanations and one action you would take for each.

<details>
<summary>Answer</summary>

1. **Cluster 3 is a doublet cluster** — doublets co-express markers from two cell types, producing a mixed profile with no unique markers. Action: run DoubletFinder or Scrublet to score doublet probability; remove high-scoring cells.

2. **Cluster 3 is biologically a transitional state** — cells in transition between two cell types will have an intermediate, non-unique expression profile. Action: run pseudotime analysis to see if Cluster 3 cells lie between two other clusters on a trajectory.

3. **Cluster 3 was over-split by the clustering algorithm** — the resolution parameter was too high, splitting a homogeneous population. Action: reduce clustering resolution (decrease Leiden resolution or k in k-means) and merge similar clusters using sub-cluster marker analysis.

</details>

---

### Level 4 — Analyse

**Q:** A colleague argues: "Since we're using log1p-normalised data for PCA, and PCA is a linear method, the non-linearity of log1p means our PCA is capturing a distorted representation of expression space." Is this argument correct? What does it miss?

<details>
<summary>Answer</summary>

The argument is partially correct but practically misguided. PCA is indeed a linear method and log1p introduces non-linearity in the transformation of raw counts. However:

1. **Variance stabilisation trumps linearity:** Raw UMI counts have variance that scales with mean (NB dispersion). PCA in raw count space would be dominated by high-expression genes whose absolute variance is large — not biologically meaningful. Log1p stabilises variance, making PCA more informative.

2. **The transformed space is still linearly analysed:** PCA is applied in log1p space, finding the principal axes of variation *within that space*. The biological interpretation is that cells that differ along PC1 have proportionally different expression of the PC1-loading genes (due to log properties).

3. **Alternatives exist:** SCTransform (Seurat) uses negative binomial regression to produce Pearson residuals as input to PCA, avoiding the log1p approximation. Benchmarks show modest improvement for some datasets. Log1p + PCA remains the community standard.

</details>

---

### Level 5 — Evaluate / Create

**Q:** Design an experiment using scRNA-seq to identify cell-type-specific drug targets for a T-cell lymphoma. Specify: (1) experimental design, (2) computational pipeline steps, (3) how you would validate the top target, (4) how you would connect to AlphaFold3 structure prediction.

<details>
<summary>Answer</summary>

**Experimental design:**
- Collect tumour biopsies from 10 patients + matched blood (control T-cells)
- Perform 10x Genomics Chromium scRNA-seq (~10,000 cells per sample)
- Include cell hashing (CITE-seq oligo barcodes) for demultiplexing and surface protein measurement
- Sequence to saturation (~50,000 reads/cell)

**Computational pipeline:**
1. QC: per-sample and per-patient batch QC; flag patient-specific outliers
2. Integration: Harmony or scVI to remove patient batch effects while preserving biological variation
3. Clustering: Leiden at multiple resolutions; annotate with CellTypist (reference atlas)
4. DE: tumour T-cells vs normal T-cells (from blood); Wilcoxon + logFC > 1, padj < 0.01
5. Pathway enrichment (Module 02/04): identify dysregulated signalling pathways
6. Target selection: druggable genes (kinases, surface receptors) that are (a) upregulated in tumour T-cells, (b) lowly expressed in other cell types (specificity), (c) co-expressed with known oncogenes

**Validation:**
- Protein expression: Western blot / flow cytometry on independent patient cohort
- Functional: CRISPR knockout in T-cell lymphoma cell line → measure proliferation, apoptosis
- In vivo: xenograft mouse model + candidate inhibitor

**AlphaFold3 connection:**
- Use AlphaFold3 (Module 07) to predict the target protein structure
- Identify potential binding pockets using FPocket or SiteMap
- Run MD simulation (Module 11) to assess pocket druggability and flexibility
- Use scRNA-seq co-expression data to identify protein interaction partners → model complex with AlphaFold3 multimer
- Design small molecule or antibody targeting the most accessible pocket

</details>

In [ ]:
# ============================================================
# Section 16 — Full Pipeline Summary Plot
# Recap figure showing the complete workflow
# ============================================================

fig = plt.figure(figsize=(18, 12))
# Visualization
fig.suptitle('scRNA-seq Analysis Pipeline — Summary\n500 Cells × 2000 Genes, 5 Cell Types',
             fontsize=14, fontweight='bold')

gs_main = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel 1: QC (library size distribution) ──────────────────
ax1 = fig.add_subplot(gs_main[0, 0])
ax1.hist(qc_filt['library_size'], bins=30, color='#3498db', edgecolor='white', linewidth=0.4)
ax1.set_title('1. QC: Library Size\n(post-filter)', fontsize=10, fontweight='bold')
ax1.set_xlabel('Total UMI per cell'); ax1.set_ylabel('Cell count')
ax1.axvline(qc_filt['library_size'].median(), color='red', linestyle='--',
            label=f'Median={qc_filt["library_size"].median():.0f}')
ax1.legend(fontsize=8)

# ── Panel 2: HVG selection ────────────────────────────────────
ax2 = fig.add_subplot(gs_main[0, 1])
ax2.scatter(gene_mean[~hvg_mask], norm_dispersion[~hvg_mask],
            s=3, alpha=0.3, color='grey')  # transition probability (complement of 0.7)
ax2.scatter(gene_mean[hvg_mask], norm_dispersion[hvg_mask],
            s=5, alpha=0.8, color='#e74c3c', label=f'HVG (n={len(hvg_genes)})')
ax2.set_title('2. HVG Selection\n(dispersion method)', fontsize=10, fontweight='bold')
ax2.set_xlabel('Mean log1p expression'); ax2.set_ylabel('Norm. dispersion')
ax2.legend(fontsize=8)

# ── Panel 3: PCA scree ────────────────────────────────────────
ax3 = fig.add_subplot(gs_main[0, 2])
ax3.bar(range(1, 16), explained_var[:15] * 100, color='steelblue', alpha=0.8)
ax3.set_title('3. PCA\nExplained Variance', fontsize=10, fontweight='bold')
ax3.set_xlabel('PC'); ax3.set_ylabel('% Variance')

# ── Panel 4: t-SNE coloured by cell type ─────────────────────
ax4 = fig.add_subplot(gs_main[1, 0])
for ct in CELL_TYPES:
    mask = labels_filt == ct
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                s=14, alpha=0.8, label=ct, color=ct_colors[ct])
ax4.set_title('4. t-SNE\nColoured by Cell Type', fontsize=10, fontweight='bold')
ax4.set_xlabel('t-SNE 1'); ax4.set_ylabel('t-SNE 2')
ax4.legend(markerscale=1.2, fontsize=7)

# ── Panel 5: Cluster vs true type ────────────────────────────
ax5 = fig.add_subplot(gs_main[1, 1])
for cl in range(N_CLUSTERS):
    mask = cluster_labels == cl
    ax5.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                s=14, alpha=0.7, color=cluster_palette[cl],  # transition probability (must sum to 1.0 with complement)
                label=f'C{cl} ({cluster_to_type[cl][:5]})')
ax5.set_title(f'5. KMeans Clusters\nAccuracy={accuracy:.1%}', fontsize=10, fontweight='bold')
ax5.set_xlabel('t-SNE 1'); ax5.set_ylabel('t-SNE 2')
ax5.legend(markerscale=1.2, fontsize=7)

# ── Panel 6: Pseudotime trajectory ───────────────────────────
ax6 = fig.add_subplot(gs_main[1, 2])
sc6 = ax6.scatter(tsne_tcell[:, 0], tsne_tcell[:, 1],
                  c=pseudotime, cmap='RdYlBu_r', s=30, alpha=0.85)
# Visualization
plt.colorbar(sc6, ax=ax6, label='Pseudotime', shrink=0.8)
ax6.set_title('6. Pseudotime\nT-cell Activation', fontsize=10, fontweight='bold')
ax6.set_xlabel('t-SNE 1'); ax6.set_ylabel('t-SNE 2')
ax6.annotate('Naive', xy=(0.05, 0.08), xycoords='axes fraction', fontsize=9,  # standard p-value significance threshold
             color='#2980b9', fontweight='bold')
ax6.annotate('Effector', xy=(0.65, 0.88), xycoords='axes fraction', fontsize=9,
             color='#c0392b', fontweight='bold')

# Visualization
plt.savefig('/tmp/scrna_pipeline_summary.png', dpi=120, bbox_inches='tight')
# Visualization
plt.show()
print('Summary figure saved to /tmp/scrna_pipeline_summary.png')

## Section 17 — What's Next

### Immediate next steps in this curriculum:

- **Module 04/01 (ML for Omics):** Take the scRNA-seq count matrix as input and train cell type classifiers (random forest, SVM, neural network). Benchmark against unsupervised clustering.
- **Module 02/04 (Pathway Enrichment):** Run GSEA on the marker gene lists from Section 9 to identify enriched biological pathways.
- **Module 07 (AlphaFold3):** Use top marker gene products (proteins) as structural biology targets.

### Real-world applications to try:

1. **Download the 10x PBMC 3k dataset** (free, 5MB): https://support.10xgenomics.com/single-cell-gene-expression/datasets  
   Apply this exact pipeline to real human blood cells. Compare your t-SNE to the 10x reference.

2. **Repeat with Scanpy** (if installed): `pip install scanpy anndata`  
   Run `sc.pp.normalize_total`, `sc.pp.log1p`, `sc.tl.leiden` — compare to our implementations.

3. **Add batch correction:** Simulate two batches by adding a systematic bias to half the cells, then correct using simple linear regression on batch dummy variables. Verify in t-SNE.

### Production pipeline checklist:

- [ ] Ambient RNA removal (SoupX / CellBender)
- [ ] Doublet detection (DoubletFinder / Scrublet)
- [ ] Batch correction (Harmony / scVI / BBKNN)
- [ ] Cell cycle scoring and regression
- [ ] Automated cell type annotation (CellTypist)
- [ ] Differential abundance testing (diffxpy / edgeR)
- [ ] RNA velocity (scVelo)

In [ ]:
# ============================================================
# Final Cell — Environment & Reproducibility Summary
# ============================================================

import sys
import sklearn
import scipy

print('=== Environment ===')
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
print(f'Sklearn : {sklearn.__version__}')
print(f'SciPy   : {scipy.__version__}')

print('\n=== Pipeline Summary ===')
print(f'Input matrix  : {count_df.shape[0]} cells x {count_df.shape[1]} genes')
print(f'After QC      : {count_filt.shape[0]} cells ({count_filt.shape[0]/count_df.shape[0]:.1%} retained)')
print(f'HVGs selected : {len(hvg_genes)}')
print(f'PCs used      : {N_PCS_FOR_TSNE} (of {N_PCS} computed)')
print(f'Clusters      : {N_CLUSTERS}')
print(f'Cluster accuracy vs true labels: {accuracy:.1%}')
print(f'T-cell pseudotime |r| with ground truth: {abs(r):.3f}')

print('\n=== Key References ===')
refs = [
    ('Macosko 2015', 'Drop-seq',          'doi:10.1016/j.cell.2015.05.002'),
    ('Wolf 2018',    'Scanpy',            'doi:10.1186/s13059-017-1382-0'),
    ('Luecken 2019', 'Best practices',    'doi:10.15252/msb.20188746'),
    ('McInnes 2018', 'UMAP',              'arxiv:1802.03426'),
    ('Trapnell 2014','Monocle pseudotime','doi:10.1038/nbt.2859'),
]
for author, topic, ref in refs:
    print(f'  {author:<16} {topic:<22} {ref}')

## Notebook Complete

**You can now:**
- Process single-cell RNA-seq data: quality control, normalisation, and clustering
- Visualise cell populations with UMAP and identify marker genes per cluster

**Where this fits:**
- Previous: 04_pathway_enrichment
- Next: 03_protein_structural_biology/01_structure_analysis — Module 03 — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes